# Annotations API Response → Parsing → Knowledge Graph

This notebook demonstrates the complete flow of biomedical annotation data:

1. **Raw API Response** (JSON-LD from Europe PMC Annotations API)
2. **Parsing** (extract entities, context, semantic URIs)
3. **RDF Conversion** (W3C Open Annotation model)
4. **Knowledge Graph** (queryable triples)

Follow along to see how mentions of diseases, chemicals, and genes from scientific papers are automatically extracted, linked to ontologies, and integrated into a knowledge graph.

## Step 1: Import Libraries

In [1]:
import json
from pprint import pprint
from rdflib import Graph, Namespace

from pyeuropepmc.processing.annotation_parser import parse_annotations
from pyeuropepmc.processing.annotations_to_rdf import annotations_to_rdf

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Step 2: Raw Annotation API Response

This is real data from the Europe PMC Annotations API. The API returns a JSON-LD array containing:

- **source**: Database source (e.g., "MED" for MEDLINE)
- **extId**: External identifier (PMID)
- **pmcid**: PMC identifier (when available)
- **annotations**: Array of extracted concept mentions with:
  - `exact`: The text span from the paper
  - `prefix` / `postfix`: Surrounding context
  - `tags`: Semantic URIs linking to ontologies (UMLS, CHEBI, GO, etc.)
  - `section`: Where in the paper it appears
  - `type`: Annotation category (Diseases, Chemicals, Gene Ontology, etc.)

In [2]:
# Real response from Europe PMC Annotations API
# This example shows annotations extracted from a ME/CFS research paper (PMID: 41444612)

RAW_API_RESPONSE = [
    {
        "source": "MED",
        "extId": "41444612",
        "pmcid": "PMC12888368",
        "annotations": [
            {
                "prefix": "and modifiers of",
                "exact": "ME/CFS",
                "postfix": "previously obscured by",
                "tags": [
                    {
                        "name": "chronic fatigue unspecified",
                        "uri": "http://linkedlifedata.com/resource/umls-concept/C0015674"
                    }
                ],
                "id": "http://europepmc.org/article/MED/41444612#europepmc-50b2a3b",
                "type": "Diseases",
                "section": "Abstract (http://purl.org/dc/terms/abstract)",
                "provider": "Europe PMC"
            },
            {
                "prefix": "regulation of glycolysis,",
                "exact": "amino acid",
                "postfix": "acid and lipid",
                "tags": [
                    {
                        "name": "amino acid",
                        "uri": "http://purl.obolibrary.org/obo/CHEBI_33709"
                    }
                ],
                "id": "http://europepmc.org/article/MED/41444612#europepmc-7e102098",
                "type": "Chemicals",
                "section": "Abstract (http://purl.org/dc/terms/abstract)",
                "provider": "Europe PMC"
            },
            {
                "prefix": "that can influence",
                "exact": "gene expression",
                "postfix": "expression profiles.",
                "tags": [
                    {
                        "name": "gene expression",
                        "uri": "http://identifiers.org/GO:0010467"
                    }
                ],
                "id": "http://europepmc.org/article/MED/41444612#europepmc-961800d8",
                "type": "Gene Ontology",
                "section": "Methods (http://purl.org/orb/Methods)",
                "provider": "Europe PMC"
            }
        ]
    }
]

print("Raw API Response (3 annotations from paper PMID:41444612):")
print("=" * 80)
pprint(RAW_API_RESPONSE, depth=2)

Raw API Response (3 annotations from paper PMID:41444612):
[{'annotations': [...],
  'extId': '41444612',
  'pmcid': 'PMC12888368',
  'source': 'MED'}]


## Step 3: Parse the API Response

The `parse_annotations()` function normalizes and structures the raw API response:
- Extracts entities with their semantic URIs (ontology links)
- Preserves contextual information (prefix, postfix, section)
- Validates the response format
- Returns structured Python dictionaries

In [3]:
# Parse the raw API response
parsed = parse_annotations(RAW_API_RESPONSE)

print("\nParsed Result Structure:")
print("=" * 80)
print(f"Valid response: {parsed['valid']}")
print(f"Entities extracted: {len(parsed['entities'])}")
print(f"Relationships: {len(parsed['relationships'])}")
print(f"Sentences: {len(parsed['sentences'])}")
print()

# Show metadata
if parsed.get('metadata'):
    print("Metadata:")
    pprint(parsed['metadata'])


Parsed Result Structure:
Valid response: True
Entities extracted: 3
Relationships: 0
Sentences: 0

Metadata:
{'page': 1, 'page_size': 3, 'source': '', 'total_count': 3, 'version': ''}


## Step 4: Inspect Extracted Entities

Each entity now includes structured fields that enable knowledge graph integration:
- `exact`: Text span from the paper
- `name`: Normalized concept name
- `id`: Semantic URI to external ontology (UMLS, CHEBI, GO, etc.)
- `annotation_category`: Type (Disease, Chemical, Gene, etc.)
- `section`: Where in the paper (Abstract, Methods, Introduction, etc.)
- `prefix` / `postfix`: Surrounding text (context preservation)

In [4]:
print("\nAll Extracted Entities:")
print("=" * 80)

for idx, entity in enumerate(parsed['entities'], 1):
    print(f"\n[Entity {idx}]")
    print(f"  Text mention: '{entity.get('exact')}'")
    print(f"  Concept name: {entity.get('name')}")
    print(f"  Category: {entity.get('annotation_category')}")
    print(f"  Semantic URI: {entity.get('id')}")
    print(f"  Section: {entity.get('section')}")
    print(f"  Context: ...{entity.get('prefix')} [{entity.get('exact')}] {entity.get('postfix')}...")


All Extracted Entities:

[Entity 1]
  Text mention: 'ME/CFS'
  Concept name: chronic fatigue unspecified
  Category: Diseases
  Semantic URI: http://linkedlifedata.com/resource/umls-concept/C0015674
  Section: Abstract (http://purl.org/dc/terms/abstract)
  Context: ...and modifiers of [ME/CFS] previously obscured by...

[Entity 2]
  Text mention: 'amino acid'
  Concept name: amino acid
  Category: Chemicals
  Semantic URI: http://purl.obolibrary.org/obo/CHEBI_33709
  Section: Abstract (http://purl.org/dc/terms/abstract)
  Context: ...regulation of glycolysis, [amino acid] acid and lipid...

[Entity 3]
  Text mention: 'gene expression'
  Concept name: gene expression
  Category: Gene Ontology
  Semantic URI: http://identifiers.org/GO:0010467
  Section: Methods (http://purl.org/orb/Methods)
  Context: ...that can influence [gene expression] expression profiles....


## Step 5: Convert to RDF (Knowledge Graph Format)

Transform parsed annotations into RDF triples following the W3C Open Annotation Data Model.

This creates:
- **oa:Annotation** instances for each mention
- **oa:hasBody** properties linking to semantic concepts (ontology URIs)
- **oa:hasTarget** properties linking back to source articles
- **nif:Phrase** for textual expressions

The result is a machine-readable knowledge graph queryable via SPARQL.

In [5]:
# Convert to RDF graph
rdf_graph = annotations_to_rdf(parsed)

print("\nRDF Graph Created:")
print("=" * 80)
print(f"Total triples: {len(rdf_graph)}")
print(f"Total namespaces: {len(list(rdf_graph.namespaces()))}")
print()

# Show namespace prefixes
print("Namespaces bound in graph:")
for prefix, namespace in rdf_graph.namespaces():
    if prefix:  # Skip default namespace
        print(f"  {prefix}: {str(namespace)[:60]}...")


RDF Graph Created:
Total triples: 57
Total namespaces: 7

Namespaces bound in graph:
  owl: http://www.w3.org/2002/07/owl#...
  rdf: http://www.w3.org/1999/02/22-rdf-syntax-ns#...
  rdfs: http://www.w3.org/2000/01/rdf-schema#...
  xsd: http://www.w3.org/2001/XMLSchema#...
  xml: http://www.w3.org/XML/1998/namespace...
  oa: http://www.w3.org/ns/oa#...
  nif: http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-co...


## Step 6: View RDF Triples

Each RDF triple is a (subject, predicate, object) statement. Together they form a queryable knowledge graph.

In [6]:
# Display all triples
print("\nRDF Triples (Sample):")
print("=" * 80)

triples = list(rdf_graph)
print(f"Total: {len(triples)} triples\n")

# Show first 10 triples
for idx, (subject, predicate, obj) in enumerate(triples[:10], 1):
    print(f"Triple {idx}:")
    print(f"  S: {str(subject)[:70]}")
    print(f"  P: {str(predicate)[:70]}")
    print(f"  O: {str(obj)[:70]}")
    print()


RDF Triples (Sample):
Total: 57 triples

Triple 1:
  S: http://example.org/annotations/3
  P: https://w3id.org/pyeuropepmc/vocab#textPostfix
  O: expression profiles.

Triple 2:
  S: http://example.org/annotations/3
  P: http://www.w3.org/ns/prov#wasDerivedFrom
  O: http://europepmc.org/article/MED/41444612

Triple 3:
  S: http://example.org/annotations/3
  P: http://www.w3.org/ns/prov#wasGeneratedBy
  O: pyeuropepmc_parser

Triple 4:
  S: http://example.org/annotations/3
  P: http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  O: http://www.w3.org/ns/oa#SpecificResource

Triple 5:
  S: http://example.org/annotations/2
  P: http://www.w3.org/ns/oa#hasTarget
  O: http://europepmc.org/article/MED/41444612

Triple 6:
  S: http://example.org/annotations/1
  P: http://www.w3.org/ns/oa#hasTarget
  O: http://europepmc.org/article/MED/41444612

Triple 7:
  S: http://example.org/annotations/1
  P: http://purl.org/dc/terms/creator
  O: Europe PMC

Triple 8:
  S: http://example.org/annotations/3
 

## Step 7: Serialize to Turtle Format

Turtle is a human-readable RDF serialization format. This shows how the knowledge graph is stored or transmitted.

In [7]:
# Serialize to Turtle format
turtle = rdf_graph.serialize(format="turtle")

print("\nRDF in Turtle Format:")
print("=" * 80)
# Show first part
lines = turtle.split('\n')
for line in lines[:40]:  # First 40 lines
    print(line)

if len(lines) > 40:
    print(f"\n... ({len(lines) - 40} more lines)")


RDF in Turtle Format:
@prefix ns1: <https://w3id.org/pyeuropepmc/vocab#> .
@prefix ns2: <http://www.w3.org/ns/prov#> .
@prefix ns3: <http://purl.org/dc/terms/> .
@prefix oa: <http://www.w3.org/ns/oa#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<http://example.org/annotations/1> a oa:Annotation,
        oa:SpecificResource,
        ns1:EntityAnnotation,
        "http" ;
    rdfs:label "chronic fatigue unspecified" ;
    ns3:creator "Europe PMC" ;
    ns3:isPartOf "Abstract (http://purl.org/dc/terms/abstract)" ;
    owl:sameAs <http://linkedlifedata.com/resource/umls-concept/C0015674> ;
    oa:hasBody "ME/CFS" ;
    oa:hasTarget "http://europepmc.org/article/MED/41444612"^^xsd:anyURI ;
    ns2:generatedAtTime "2026-04-24T18:54:57.368802" ;
    ns2:wasDerivedFrom <http://europepmc.org/article/MED/41444612> ;
    ns2:wasGeneratedBy "pyeuropepmc_parser" ;
    ns1:annotationTyp

## Step 8: Query the Knowledge Graph

Use SPARQL to query the RDF graph. Let's find all concepts and their semantic URIs.

In [8]:
# Query to find all subjects and their types
query_results = list(rdf_graph.subjects())

print("\nUnique Subject URIs in Graph:")
print("=" * 80)
print(f"Total: {len(set(query_results))}\n")

for subject in set(query_results):
    print(f"  {str(subject)[:70]}")


Unique Subject URIs in Graph:
Total: 3

  http://example.org/annotations/1
  http://example.org/annotations/3
  http://example.org/annotations/2


## Step 9: Summary - Complete Flow

### The Annotation Pipeline:

```
┌─────────────────────────────────────────────────────────────────┐
│  Europe PMC Annotations API                                     │
│  (Raw JSON-LD response)                                         │
└────────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────────┐
│  parse_annotations()                                            │
│  Extract entities, relationships, metadata                      │
└────────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────────┐
│  Structured Python Data                                         │
│  - entities: concept mentions with ontology URIs                │
│  - context: prefix, postfix, section                           │
│  - semantic: links to UMLS, CHEBI, GO, etc.                    │
└────────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────────┐
│  annotations_to_rdf()                                           │
│  Convert to W3C Open Annotation model                           │
└────────────────────────────────────────────────────────────────┘
,
│  - oa:Annotation instances                                      │
│  - oa:hasBody → semantic concepts                               │
│  - oa:hasTarget → source articles                               │
│  - Rich semantic triples                                        │
└────────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────────┐
│  Knowledge Graph                                                │
│  - SPARQL-queryable                                             │
│  - Interoperable (linked to standard ontologies)                │
│  - Context-preserving (textual spans, sections)                 │
└────────────────────────────────────────────────────────────────┘
```

In [9]:
# Final summary
print("\n" + "=" * 80)
print("SUMMARY: Raw API → Parsed Data → RDF Knowledge Graph")
print("=" * 80)
print()
print("📥 Input: 1 API response document")
print(f"   - 3 annotations (ME/CFS, amino acid, gene expression)")
print()
print("🔍 Parsed: Extracted entities")
print(f"   - {len(parsed['entities'])} entities with semantic URIs")
print(f"   - Linked to UMLS, CHEBI, and Gene Ontology")
print(f"   - Context preserved (prefix/postfix/section)")
print()
print("📊 Knowledge Graph: RDF triples")
print(f"   - {len(rdf_graph)} triples created")
print(f"   - W3C Open Annotation model")
print(f"   - SPARQL-queryable")
print()
print("✓ Result: Automated biomedical concept extraction from literature")
print("           with full semantic interoperability and context preservation!")


SUMMARY: Raw API → Parsed Data → RDF Knowledge Graph

📥 Input: 1 API response document
   - 3 annotations (ME/CFS, amino acid, gene expression)

🔍 Parsed: Extracted entities
   - 3 entities with semantic URIs
   - Linked to UMLS, CHEBI, and Gene Ontology
   - Context preserved (prefix/postfix/section)

📊 Knowledge Graph: RDF triples
   - 57 triples created
   - W3C Open Annotation model
   - SPARQL-queryable

✓ Result: Automated biomedical concept extraction from literature
           with full semantic interoperability and context preservation!


# Annotations API to Knowledge Graph: Complete Demo

This notebook demonstrates the complete flow of annotation data from the Europe PMC Annotations API:

1. **Raw API Response** - What the API returns (JSON-LD with semantic annotations)
2. **Parsing** - Extract structured entities, relationships, and context
3. **Enrichment** - Add semantic URIs, categories, and contextual information
4. **RDF Conversion** - Transform to W3C Open Annotation model triples
5. **KG Integration** - Load into a knowledge graph

This shows how biomedical concepts (diseases, chemicals, genes) from literature are automatically extracted, linked to ontologies, and made queryable in a knowledge graph.

## 1. Import Required Libraries

In [10]:
import json
from pprint import pprint
from rdflib import Graph, Namespace, URIRef, Literal

from pyeuropepmc.processing.annotation_parser import parse_annotations
from pyeuropepmc.processing.annotations_to_rdf import annotations_to_rdf

print("✓ All imports successful")

✓ All imports successful


## 2. Raw Annotation API Response

This is what the Europe PMC Annotations API returns. It's a JSON-LD response containing:
- `source`: Database source (MED = MEDLINE)
- `extId`: External ID (PMID or other identifier)
- `pmcid`: PMC ID (when available)
- `annotations`: Array of annotated mentions with semantic links

In [11]:
# Real response from Europe PMC Annotations API
# This shows annotations from a ME/CFS research paper

RAW_API_RESPONSE = [
    {
        "source": "MED",
        "extId": "41444612",
        "pmcid": "PMC12888368",
        "annotations": [
            # Annotation 1: Disease mention
            {
                "prefix": "and modifiers of",
                "exact": "ME/CFS",
                "postfix": "previously obscured by",
                "tags": [
                    {
                        "name": "chronic fatigue unspecified",
                        "uri": "http://linkedlifedata.com/resource/umls-concept/C0015674"
                    }
                ],
                "id": "http://europepmc.org/article/MED/41444612#europepmc-50b2a3b10e3cfb3a942c6acb716dbf59",
                "type": "Diseases",
                "section": "Abstract (http://purl.org/dc/terms/abstract)",
                "provider": "Europe PMC"
            },
            # Annotation 2: Chemical mention
            {
                "prefix": "regulation of glycolysis,",
                "exact": "amino acid",
                "postfix": "acid and lipid",
                "tags": [
                    {
                        "name": "amino acid",
                        "uri": "http://purl.obolibrary.org/obo/CHEBI_33709"
                    }
                ],
                "id": "http://europepmc.org/article/MED/41444612#europepmc-7e10209829c819602f822cb9fb811974",
                "type": "Chemicals",
                "section": "Abstract (http://purl.org/dc/terms/abstract)",
                "provider": "Europe PMC"
            },
            # Annotation 3: Gene Ontology mention
            {
                "prefix": "that can influence",
                "exact": "gene expression",
                "postfix": "expression profiles.",
                "tags": [
                    {
                        "name": "gene expression",
                        "uri": "http://identifiers.org/GO:0010467"
                    }
                ],
                "id": "http://europepmc.org/article/MED/41444612#europepmc-961800d8d3422cb6fa1d6d47d5a7f6fe",
                "type": "Gene Ontology",
                "section": "Methods (http://purl.org/orb/Methods)",
                "provider": "Europe PMC"
            }
        ]
    }
]

print("Raw API Response (Pretty-printed):")
print("=" * 80)
pprint(RAW_API_RESPONSE)

Raw API Response (Pretty-printed):
[{'annotations': [{'exact': 'ME/CFS',
                   'id': 'http://europepmc.org/article/MED/41444612#europepmc-50b2a3b10e3cfb3a942c6acb716dbf59',
                   'postfix': 'previously obscured by',
                   'prefix': 'and modifiers of',
                   'provider': 'Europe PMC',
                   'section': 'Abstract (http://purl.org/dc/terms/abstract)',
                   'tags': [{'name': 'chronic fatigue unspecified',
                             'uri': 'http://linkedlifedata.com/resource/umls-concept/C0015674'}],
                   'type': 'Diseases'},
                  {'exact': 'amino acid',
                   'id': 'http://europepmc.org/article/MED/41444612#europepmc-7e10209829c819602f822cb9fb811974',
                   'postfix': 'acid and lipid',
                   'prefix': 'regulation of glycolysis,',
                   'provider': 'Europe PMC',
                   'section': 'Abstract (http://purl.org/dc/terms/abstract

## 3. Parse and Inspect the Raw Response

The `parse_annotations()` function extracts and structures the raw API response:
- Normalizes the JSON-LD format
- Extracts entities with their semantic URIs
- Preserves contextual information (prefix, postfix, section)
- Links to external ontologies (UMLS, CHEBI, Gene Ontology)

In [12]:
# Step 1: Parse the raw API response
parsed_result = parse_annotations(RAW_API_RESPONSE)

print("Parsed Result Structure:")
print("=" * 80)
print(f"Valid: {parsed_result['valid']}")
print(f"Number of entities: {len(parsed_result['entities'])}")
print(f"Number of sentences: {len(parsed_result['sentences'])}")
print(f"Number of relationships: {len(parsed_result['relationships'])}")
print()

# Show the first entity in detail
if parsed_result['entities']:
    print("\nFirst Entity (Full Structure):")
    print("-" * 80)
    pprint(parsed_result['entities'][0], width=120)

Parsed Result Structure:
Valid: True
Number of entities: 3
Number of sentences: 0
Number of relationships: 0


First Entity (Full Structure):
--------------------------------------------------------------------------------
{'annotation_category': 'Diseases',
 'annotation_id': 'http://europepmc.org/article/MED/41444612#europepmc-50b2a3b10e3cfb3a942c6acb716dbf59',
 'article_id': 'PMC12888368',
 'article_uri': 'http://europepmc.org/article/MED/41444612',
 'confidence': None,
 'exact': 'ME/CFS',
 'id': 'http://linkedlifedata.com/resource/umls-concept/C0015674',
 'name': 'chronic fatigue unspecified',
 'postfix': 'previously obscured by',
 'prefix': 'and modifiers of',
 'provider': 'Europe PMC',
 'section': 'Abstract (http://purl.org/dc/terms/abstract)',
 'type': 'http'}


## 4. Display All Extracted Entities

Show the structured entities extracted from the API response. Each entity includes:
- **exact**: The text mention from the paper
- **name**: The normalized concept name
- **id**: Semantic URI linking to an external ontology
- **annotation_category**: Type classification (Disease, Chemical, Gene, etc.)
- **prefix/postfix**: Surrounding context for understanding meaning
- **section**: Where in the paper it appears (Abstract, Methods, etc.)

In [13]:
print("\nAll Extracted Entities:")
print("=" * 80)

for idx, entity in enumerate(parsed_result['entities'], 1):
    print(f"\nEntity {idx}: {entity.get('name', 'Unknown')}")
    print(f"  Text mention: '{entity.get('exact')}'")
    print(f"  Type: {entity.get('annotation_category')}")
    print(f"  Semantic URI: {entity.get('id')}")
    print(f"  Context: [{entity.get('prefix')}] **{entity.get('exact')}** [{entity.get('postfix')}]")
    print(f"  Section: {entity.get('section')}")
    print(f"  Provider: {entity.get('provider')}")


All Extracted Entities:

Entity 1: chronic fatigue unspecified
  Text mention: 'ME/CFS'
  Type: Diseases
  Semantic URI: http://linkedlifedata.com/resource/umls-concept/C0015674
  Context: [and modifiers of] **ME/CFS** [previously obscured by]
  Section: Abstract (http://purl.org/dc/terms/abstract)
  Provider: Europe PMC

Entity 2: amino acid
  Text mention: 'amino acid'
  Type: Chemicals
  Semantic URI: http://purl.obolibrary.org/obo/CHEBI_33709
  Context: [regulation of glycolysis,] **amino acid** [acid and lipid]
  Section: Abstract (http://purl.org/dc/terms/abstract)
  Provider: Europe PMC

Entity 3: gene expression
  Text mention: 'gene expression'
  Type: Gene Ontology
  Semantic URI: http://identifiers.org/GO:0010467
  Context: [that can influence] **gene expression** [expression profiles.]
  Section: Methods (http://purl.org/orb/Methods)
  Provider: Europe PMC


## 5. Transform to RDF (W3C Open Annotation Model)

Convert the parsed annotations into RDF triples following the W3C Open Annotation Data Model. This creates:
- **oa:Annotation** instances for each mention
- **oa:hasBody** links to semantic concepts
- **oa:hasTarget** links to source articles
- **nif:Phrase** for textual expressions

The result is a queryable knowledge graph.

In [14]:
# Step 2: Convert to RDF graph
rdf_graph = annotations_to_rdf(parsed_result)

print(f"RDF Graph Statistics:")
print("=" * 80)
print(f"Total triples: {len(rdf_graph)}")
print(f"Namespaces: {len(rdf_graph.namespaces())}")
print()

# Show namespace bindings
print("Bound Namespaces:")
for prefix, namespace in rdf_graph.namespaces():
    print(f"  {prefix}: {namespace}")

RDF Graph Statistics:
Total triples: 57


TypeError: object of type 'generator' has no len()

## 6. Inspect the RDF Graph

Query and display the RDF triples. Each triple follows the (subject, predicate, object) model.

In [ ]:
# Display all triples in the RDF graph
print("\nRDF Triples (Subject, Predicate, Object):")
print("=" * 80)

for idx, (s, p, o) in enumerate(rdf_graph, 1):
    print(f"\n[{idx}] Triple:")
    print(f"  Subject:   {s}")
    print(f"  Predicate: {p}")
    print(f"  Object:    {o}")

## 7. Serialize RDF to Turtle Format

Turtle (TTL) is a human-readable RDF format. This shows how the RDF graph would be stored or transmitted.

In [ ]:
# Serialize to Turtle format
turtle_output = rdf_graph.serialize(format="turtle")

print("\nRDF Graph in Turtle Format:")
print("=" * 80)
print(turtle_output[:2000])  # Print first 2000 chars

if len(turtle_output) > 2000:
    print(f"\n... ({len(turtle_output) - 2000} more characters)")

## 8. Knowledge Graph Queries

Query the RDF graph using SPARQL to find specific patterns and concepts.

In [ ]:
# Query 1: Find all entity mentions and their semantic URIs
query_entities = """
SELECT ?annotation ?text ?concept ?label WHERE {
    ?annotation ?p ?o .
    OPTIONAL { ?annotation <http://www.w3.org/ns/oa#hasBody> ?concept . }
    OPTIONAL { ?annotation <https://w3id.org/pyeuropepmc/vocab#textPrefix> ?text . }
    OPTIONAL { ?concept <http://www.w3.org/2000/01/rdf-schema#label> ?label . }
}

## 9. Summary: From API to KG

### Flow Summary:

1. **API Response** → Raw JSON-LD with semantic annotations
2. **Parsing** → Extract entities, relationships, contextual metadata
3. **Enrichment** → Link to ontologies (UMLS, CHEBI, Gene Ontology, etc.)
4. **RDF Conversion** → Transform to W3C Open Annotation model
5. **KG Integration** → Triples queryable via SPARQL

### What You Get:

- ✓ Automatically extracted biomedical concepts from literature
- ✓ Linked to international ontologies for interoperability
- ✓ Contextual information preserved (where in text, sentence, section)
- ✓ Machine-readable semantic relationships
- ✓ Queryable knowledge graph ready for analysis

In [ ]:
# Summary statistics
print("\n" + "=" * 80)
print("SUMMARY: API Response → Parsed Data → RDF KG")
print("=" * 80)
print(f"\n✓ Raw API response: 1 document with {len(RAW_API_RESPONSE[0]['annotations'])} annotations")
print(f"✓ Parsed entities: {len(parsed_result['entities'])} extracted")
print(f"✓ RDF triples: {len(rdf_graph)} triples in knowledge graph")
print(f"✓ Ontologies linked: UMLS, CHEBI, Gene Ontology (GO)")
print(f"\n✓ Result: Queryable biomedical knowledge graph ready for downstream analysis!")